# 03 · ML, projection 2030 et suggestions

In [ ]:
%pip install -q geopandas xlrd openpyxl scikit-learn requests pyarrow

In [ ]:
ROOT = "/lakehouse/default"
BUDGET_EUR = 500000
HORIZON_ANS = 5
TVA = 0.20
VELOS_PAR_STATION = 15
DENSITE_MIN_VLS = 7000
RAYON_M = 350
SEUIL_POP_DIFFUSION = 200
W_CARBONE = 0.5
W_EQUITE = 0.5

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_validate, cross_val_predict

OUT = Path(ROOT) / "Files/hackaton/out"
OUT.mkdir(parents=True, exist_ok=True)
read = lambda name: spark.read.table(name).toPandas()

def save(df, name):
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(name, len(df))

silver = read("silver_iris")
villes = read("bronze_villes")
om = read("bronze_omphale")

In [ ]:
def features(df, year):
    return pd.DataFrame({
        "log_pop": np.log(df[f"pop_{year}"]),
        "log_densite": np.log(df[f"densite_{year}"]),
        "part_0014": df[f"part_0014_{year}"],
        "part_1529": df[f"part_1529_{year}"],
        "part_60p": df[f"part_60p_{year}"],
        "lgt_par_hab_an": df[f"lgt_par_hab_an_{year}"],
    })

train = silver[(silver.pop_2015 > 0) & (silver.pop_2021 > 0)]
X, y = features(train, 2015), train.tx_annuel_15_21
models = {"baseline_moyenne": DummyRegressor(), "regression_lineaire": LinearRegression(), "gradient_boosting": GradientBoostingRegressor(random_state=0)}
splits = {"kfold_5": (KFold(5, shuffle=True, random_state=0), None), "par_ville": (GroupKFold(5), train.ville)}

metrics = {"n_iris_train": len(train), "cible": "taux de croissance annuel 2015-2021", "modeles": {}}
for m, model in models.items():
    metrics["modeles"][m] = {}
    for s, (cv, groups) in splits.items():
        r = cross_validate(model, X, y, cv=cv, groups=groups, scoring=["r2", "neg_mean_absolute_error"])
        metrics["modeles"][m][s] = {"r2": round(r["test_r2"].mean(), 4), "mae": round(-r["test_neg_mean_absolute_error"].mean(), 5)}
best = min(metrics["modeles"], key=lambda m: metrics["modeles"][m]["kfold_5"]["mae"])
metrics["validation_jury"] = {"part_iris_en_hausse": round((y > 0).mean(), 4), "croissance_moyenne": round(y.mean(), 5)}
for s, (cv, groups) in splits.items():
    p = cross_val_predict(models[best], X, y, cv=cv, groups=groups)
    top, vrai_top = p >= np.quantile(p, 0.8), y >= np.quantile(y, 0.8)
    metrics["validation_jury"][s] = {"bon_sens_tendance": round((np.sign(p) == np.sign(y)).mean(), 4), "top20_part_en_hausse": round((y[top] > 0).mean(), 4), "top20_croissance_moyenne": round(y[top].mean(), 5),
        "f1_hausse": round(f1_score(y > 0, p > 0), 4), "f1_macro_hausse": round(f1_score(y > 0, p > 0, average="macro"), 4), "f1_hausse_baseline": round(f1_score(y > 0, np.ones(len(y), bool)), 4),
        "precision_top20": round(precision_score(vrai_top, top), 4), "rappel_top20": round(recall_score(vrai_top, top), 4), "f1_top20": round(f1_score(vrai_top, top), 4)}
metrics["modele_retenu"] = best
model = models[best].fit(X, y)
if best == "gradient_boosting":
    metrics["importance_variables"] = dict(zip(X.columns, model.feature_importances_.round(3).tolist()))
(OUT / "ml_metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
pd.DataFrame({m: {s: v["mae"] for s, v in d.items()} for m, d in metrics["modeles"].items()})

In [ ]:
om["tx_omphale_21_30"] = (om.POP_2030 / om.POP_2021) ** (1 / 9) - 1
gold = silver[silver.pop_2021 > 0].merge(villes[["ville", "dep"]], on="ville").merge(om[["DEP", "tx_omphale_21_30"]], left_on="dep", right_on="DEP")
gold["tx_modele"] = model.predict(features(gold, 2021))
moy = gold.groupby("dep").apply(lambda g: np.average(g.tx_modele, weights=g.pop_2021)).rename("tx_modele_moyen_dep")
gold = gold.merge(moy, left_on="dep", right_index=True)
gold["tx_proj_21_30"] = gold.tx_modele - gold.tx_modele_moyen_dep + gold.tx_omphale_21_30
gold["pop_2030_proj"] = (gold.pop_2021 * (1 + gold.tx_proj_21_30) ** 9).round()
gold["delta_pop_21_30"] = gold.pop_2030_proj - gold.pop_2021
gold["evolution_predite"] = np.where(gold.tx_proj_21_30 > 0, "croissance", "décroissance")
gold["rang_motorisation"] = gold.groupby("ville").part_menages_voiture.rank(pct=True)
gold["equite"] = np.maximum(gold.groupby("ville").taux_pauvrete.rank(pct=True).fillna(0), gold.qpv.fillna(False).astype(float))
gold = gold.drop(columns=["DEP"])
save(gold, "gold_iris")

In [ ]:
couts = pd.DataFrame([
    ["citiz", 7250 * (1 + TVA), 5160 * (1 + TVA), "Fiche Citiz (Dromolib 2025) : investissement 7 250 € HT par station d'une voiture, fonctionnement 5 160 € HT/an avec voiture électrique fournie par Citiz, amortissement inclus ; TVA 20 %", "https://dromolib.fr/wp-content/uploads/2025/11/FICHE_Citiz_Autopartage.pdf"],
    ["arret_bus", 15000, 0, "Cerema, panorama de 55 SD'AP : coût moyen d'un arrêt 15 000 € TTC ; aucun coût public d'exploitation par arrêt trouvé", "https://www.cerema.fr/system/files/documents/2018/02/infographie_panorama_55_dossiers_sd_ap_cle53c7d9.pdf"],
    ["velo", 0, 2194 * VELOS_PAR_STATION, "ADEME, Étude d'évaluation sur les services vélos : 2 194 €/vélo/an tout compris (agglo > 500 000 hab), 12 à 15 vélos par station", "https://www.actu-environnement.com/media/pdf/dossiers/808-ademe-services-velo.pdf"],
], columns=["type", "cout_invest_ttc_eur", "cout_fonct_an_ttc_eur", "source", "url"])
couts["cout_horizon_eur"] = couts.cout_invest_ttc_eur + HORIZON_ANS * couts.cout_fonct_an_ttc_eur
save(couts, "gold_couts")

In [ ]:
villes_vls = set(read("bronze_velo").ville)
base = gold[gold.pop_2021 >= SEUIL_POP_DIFFUSION]
cand = []
for t in couts.itertuples():
    c = base[base[f"gain_ind_{t.type}"] > 0].copy()
    if t.type == "velo":
        c = c[c.ville.isin(villes_vls) & (c.densite_2021 >= DENSITE_MIN_VLS)]
    c["type"], c["cout_eur"] = t.type, t.cout_horizon_eur
    c["gain_pop_2030"] = (c[f"gain_ind_{t.type}"] / c.ind_carreaux * c.pop_2030_proj).round()
    c["part_pop_desservie"], c["dist_pond_m"] = c[f"part_pop_desservie_{t.type}"], c[f"dist_pond_{t.type}_m"]
    c["site_lat"], c["site_lon"] = c[f"site_lat_{t.type}"], c[f"site_lon_{t.type}"]
    cand.append(c)
cand = pd.concat(cand)
cand["score"] = cand.gain_pop_2030 * (1 + W_CARBONE * cand.rang_motorisation.fillna(0) + W_EQUITE * cand.equite)
cand["score_par_keur"] = cand.score / cand.cout_eur * 1000
cand = cand.sort_values("score_par_keur", ascending=False)
cand["meilleur_type_iris"] = ~cand.duplicated("code_iris")
best = cand[cand.meilleur_type_iris].copy()
best["cout_cumule_eur"] = best.groupby("ville").cout_eur.cumsum()
cand = cand.merge(best[["code_iris", "type", "cout_cumule_eur"]], on=["code_iris", "type"], how="left")
cand["recommande"] = cand.cout_cumule_eur <= BUDGET_EUR
cand["rang_ville"] = cand.groupby("ville").cumcount() + 1
cand["installation"] = cand.type.map({"citiz": "Station Citiz", "arret_bus": "Arrêt de bus", "velo": "Station vélo"})
cand["pop_2021"] = cand.pop_2021.round()
cand["pourquoi"] = cand.apply(lambda r: (f"Besoin d'offre : {r.gain_pop_2030:_.0f} hab. (2030) à plus de {RAYON_M} m d'une offre {r.installation.lower()} ; évolution prédite {r.tx_proj_21_30:+.2%}/an ; "
    f"{r.part_menages_voiture:.0%} des ménages motorisés ; taux de pauvreté {r.taux_pauvrete:.0%}{' ; QPV' if r.qpv else ''} ; "
    f"{r.lgt_autorises_2021_2025:_.0f} logements autorisés 2021-2025 ; coût {r.cout_eur:_.0f} € TTC sur {HORIZON_ANS} ans").replace("_", " "), axis=1)
sugg = cand[["ville", "rang_ville", "recommande", "meilleur_type_iris", "type", "installation", "code_iris", "nom_iris", "nom_commune", "site_lat", "site_lon", "pop_2021", "pop_2030_proj", "gain_pop_2030", "tx_proj_21_30", "part_pop_desservie", "dist_pond_m", "part_menages_voiture", "taux_pauvrete", "niveau_vie_moyen", "qpv", "lgt_autorises_2021_2025", "densite_2021", "cout_eur", "cout_cumule_eur", "score", "score_par_keur", "pourquoi"]]
save(sugg, "gold_suggestions")
sugg[sugg.recommande].to_csv(OUT / "top_opportunites.csv", index=False)
sugg[sugg.recommande].groupby("ville").head(3)[["ville", "installation", "nom_commune", "nom_iris", "pourquoi"]]